In [2]:
from langchain_community.document_loaders import PyPDFLoader

document = PyPDFLoader(file_path="./documents/cricket-laws-2019.pdf")
documents = document.load()

Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 39 0 (offset 0)
Ignoring wrong pointing object 390 0 (offset 0)
Ignoring wrong pointing object 392 0 (offset 0)


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n \n", "\n", ". ", " ", ""])
chunks = text_splitter.split_documents(documents=documents)

In [3]:
from collections import defaultdict
tagging_dict = defaultdict(int)

for chunk in chunks:
    source = chunk.metadata.get("source")
    page = chunk.metadata.get("page")
    block = f"{source}:{page}"
    chunk.metadata["id"] = "{block}:{chunk_id}".format(block=block, chunk_id=tagging_dict[block])
    tagging_dict[block] += 1


In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")
vector_db = Chroma(collection_name="cricket_meta", embedding_function=embeddings, persist_directory="./chroma_data/chroma_default_db")

In [14]:
def get_context(user_query: str) -> dict:
    """
    This function takes a user query as input and returns a dictionary containing the context variable and the user question.
    """
    # not scalabale and customizable for production.
    # results = vector_db.similarity_search_with_score(query=user_query, k=5)

    # use retrievers to do the job
    retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.8})
    docs = retriever.invoke(user_query)

    context_var = ""
    for doc in docs:
        context_var += doc.page_content + "\n"

    return {
        "context_var": context_var,
        "user_question": user_query
    }

In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.1:latest", temperature=0.6)
parser = StrOutputParser()
prompt = PromptTemplate(template="""
Answer the question based only on the following context. if you don't know the answer say 'I don't know'.

<context>
{context_var}
</context>

Answer the question based on the above context: {user_question}                       
""", input_variables=["context_var", "user_question"])

In [19]:
chain = get_context | prompt | llm | parser

In [21]:
response = chain.invoke(input="How many balls in an over?")

In [22]:
response

'6 balls.'

In [23]:
user_query = "What does innings mean and how many innings are there in a test match?"
response = chain.invoke(input=user_query)

In [24]:
response

'According to the context, an "innings" refers to a period of batting by a team, during which they try to score as many runs as possible. In a two-innings match (such as a Test match), each team takes their innings alternately, except in certain specific cases (e.g. the follow-on or forfeiture of an innings). Therefore, there are typically 2 innings in a Test match, one for each team.'